# Iris Flower Classification

## VedGrow Generative AI Internship Task

### Objective
Train and evaluate classification models on the classic Iris dataset.

### Requirements Covered
- Load and explore the Iris dataset using EDA and visualizations
- Train at least 3 models: KNN, Decision Tree, SVM
- Compare accuracy, precision, recall, and F1-score
- Show confusion matrix visualization for each model
- Predict species for custom input values

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import warnings
warnings.filterwarnings("ignore")

sns.set(style="whitegrid")

## 2. Load the Iris Dataset

In [ ]:
iris = load_iris()

df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["target"] = iris.target
df["species"] = df["target"].map({
    0: "setosa",
    1: "versicolor",
    2: "virginica"
})

df.head()

## 3. Basic Dataset Information

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns)

print("\nDataset information:")
df.info()

In [ ]:
print("Statistical Summary:")
df.describe()

In [ ]:
print("Missing Values:")
df.isnull().sum()

In [ ]:
print("Species Count:")
df["species"].value_counts()

## 4. Exploratory Data Analysis with Visualizations

### 4.1 Species Count Plot

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="species")
plt.title("Count of Iris Species")
plt.xlabel("Species")
plt.ylabel("Count")
plt.show()

### 4.2 Pair Plot

In [ ]:
sns.pairplot(df, hue="species", vars=iris.feature_names)
plt.suptitle("Pair Plot of Iris Features", y=1.02)
plt.show()

### 4.3 Correlation Heatmap

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[iris.feature_names].corr(), annot=True, cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.show()

### 4.4 Box Plots for Feature Distribution

In [ ]:
for feature in iris.feature_names:
    plt.figure(figsize=(7, 5))
    sns.boxplot(data=df, x="species", y=feature)
    plt.title(f"{feature} Distribution by Species")
    plt.xlabel("Species")
    plt.ylabel(feature)
    plt.show()

## 5. Prepare Data for Model Training

In [ ]:
X = df[iris.feature_names]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

### Feature Scaling

KNN and SVM perform better when features are scaled.  
Decision Tree does not require scaling, but using the same scaled data keeps the comparison simple.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 6. Train Classification Models

In [ ]:
models = {
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "SVM": SVC(kernel="rbf", random_state=42)
}

trained_models = {}

for model_name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained_models[model_name] = model
    print(f"{model_name} model trained successfully.")

## 7. Evaluate and Compare Models

In [ ]:
results = []

for model_name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted")
    recall = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average="weighted")

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

results_df = pd.DataFrame(results)
results_df

### Classification Report for Each Model

In [ ]:
for model_name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)

    print("=" * 60)
    print(f"Classification Report for {model_name}")
    print("=" * 60)
    print(classification_report(y_test, y_pred, target_names=iris.target_names))

## 8. Confusion Matrix Visualization for Each Model

In [ ]:
for model_name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=iris.target_names,
        yticklabels=iris.target_names
    )
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted Species")
    plt.ylabel("Actual Species")
    plt.show()

## 9. Visual Comparison of Model Performance

In [ ]:
results_melted = results_df.melt(id_vars="Model", var_name="Metric", value_name="Score")

plt.figure(figsize=(10, 6))
sns.barplot(data=results_melted, x="Model", y="Score", hue="Metric")
plt.title("Model Performance Comparison")
plt.ylim(0, 1.1)
plt.ylabel("Score")
plt.show()

## 10. Predict Species for Custom Input Values

In [ ]:
def predict_iris_species(sepal_length, sepal_width, petal_length, petal_width, model_name="SVM"):
    """
    Predict Iris species using the selected trained model.

    Parameters:
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float
    model_name: str, one of "KNN", "Decision Tree", or "SVM"

    Returns:
    Predicted species name
    """

    if model_name not in trained_models:
        raise ValueError("Invalid model name. Choose from: KNN, Decision Tree, SVM")

    custom_input = pd.DataFrame(
        [[sepal_length, sepal_width, petal_length, petal_width]],
        columns=iris.feature_names
    )

    custom_input_scaled = scaler.transform(custom_input)
    prediction = trained_models[model_name].predict(custom_input_scaled)[0]
    species_name = iris.target_names[prediction]

    return species_name


# Example custom prediction
custom_prediction = predict_iris_species(
    sepal_length=5.1,
    sepal_width=3.5,
    petal_length=1.4,
    petal_width=0.2,
    model_name="SVM"
)

print("Predicted Iris Species:", custom_prediction)

## 11. Try More Custom Inputs

In [ ]:
test_samples = [
    [5.1, 3.5, 1.4, 0.2],
    [6.0, 2.9, 4.5, 1.5],
    [6.7, 3.0, 5.2, 2.3]
]

for sample in test_samples:
    prediction = predict_iris_species(
        sepal_length=sample[0],
        sepal_width=sample[1],
        petal_length=sample[2],
        petal_width=sample[3],
        model_name="SVM"
    )

    print(f"Input values: {sample} --> Predicted species: {prediction}")

## 12. Conclusion

In this project, three classification models were trained and evaluated on the Iris dataset:

- KNN
- Decision Tree
- SVM

The models were compared using accuracy, precision, recall, and F1-score.  
Confusion matrices were also visualized for each model.

The trained model can successfully predict Iris species using custom flower measurement inputs.